# CRIVO — Fase 4: porta conversacional do realizador 17M

O realizador da fase 3 passou nos gates do teste cego. Aqui ele conversa **dentro do CRIVO de verdade**: o CRIVO decide (memória, fichas com fonte, recusas) e o 17M só dá a voz. Cada frase dele passa pela mesma conferência do blind e pela revisão; se falhar, fica o texto de sempre.

**Conjunto congelado:** 28 conversas, 45 turnos, escritas antes de existir qualquer saída do realizador (SHA conferido). Os critérios valem igual para o CRIVO de hoje e para o CRIVO com a voz do 17M.

**Gates:** nenhuma regressão; a conferência aprova a voz do 17M em ≥ 60% dos turnos em que ele pode falar; a revisão não precisa corrigir nenhuma frase dele. **Mesmo passando, nada vai para o chat** até o criador do projeto ler o lado a lado.

**Não precisa de GPU** (roda no processador): Ambiente de execução → Executar tudo. Leva poucos minutos. O resultado fica em `MyDrive/CRIVO/fase4-porta/`.


In [ ]:
# Célula 0 — Drive e pesos do realizador (fase 3)
import os, sys, json, shutil, subprocess, hashlib
from pathlib import Path
from datetime import datetime
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-17m-fase3").is_dir()), MEU / "CRIVO")
execucoes = sorted((CRIVO / "transformer-17m-fase3").glob("fase3_*/pesos.pt"))
if not execucoes:
    raise SystemExit("ERRO: não achei CRIVO/transformer-17m-fase3/fase3_*/pesos.pt. Crie o atalho da pasta CRIVO em Meu Drive.")
PESOS = execucoes[-1].parent
print("Pasta do CRIVO:", CRIVO, "| realizador:", PESOS, flush=True)
LOCAL = Path("/content/realizador")
shutil.rmtree(LOCAL, ignore_errors=True)
shutil.copytree(PESOS, LOCAL)
SAIDA = CRIVO / "fase4-porta" / datetime.now().strftime("execucao_%Y%m%d_%H%M%S")
SAIDA.mkdir(parents=True, exist_ok=True)


In [ ]:
# Célula 1 — código do CRIVO (branch do experimento)
REPO = Path("/content/CRIVO")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("commit", COMMIT, flush=True)


In [ ]:
# Célula 2 — porta conversacional: o mesmo CRIVO com e sem a voz do 17M
TMP = Path("/content/porta")
cmd = [sys.executable, "-u", "experimentos/fase4_porta_20261013/avaliar_porta.py", "--saida", str(TMP),
       "--realizador", str(LOCAL)]
with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for linha in p.stdout:
        if "POST /api" not in linha:
            print(linha, end="", flush=True)
if p.returncode:
    raise SystemExit("a avaliação falhou (código %d)" % p.returncode)
for nome in ("resultado_porta.json", "lado_a_lado.md"):
    shutil.copyfile(TMP / nome, SAIDA / nome)
(SAIDA / "commit.txt").write_text(COMMIT + "\n")
print("\nSALVO EM:", SAIDA, flush=True)


In [ ]:
# Célula 3 — lado a lado (leia e me diga qual soa mais como o CRIVO)
print((TMP / "lado_a_lado.md").read_text(encoding="utf-8"))
